In [1]:
# Downloading the dataset for this analysis
import kagglehub

# Download latest version
path = kagglehub.dataset_download("gandpablo/news-articles-for-political-bias-classification")

print("Path to dataset files:", path)

C:\Users\Owner\Downloads\Datamining_final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\Owner\.cache\kagglehub\datasets\gandpablo\news-articles-for-political-bias-classification\versions\1


In [84]:
import pandas as pd
import numpy as np
import os
os.environ["GIT_PYTHON_REFRESH"] = "quiet"
import warnings
import spacy
import scattertext as st
from pprint import pprint

warnings.filterwarnings("ignore")


In [4]:
df = pd.read_csv(path + "/bias_clean.csv")
print(df.head())

                                                 url         topic  \
0  https://www.npr.org/2021/01/01/952336030/what-...  general-news   
1  https://www.cbsnews.com/news/new-years-eve-gat...  general-news   
2  https://nypost.com/2020/12/31/heres-how-countr...  general-news   
3  https://www.theguardian.com/world/2020/dec/31/...   coronavirus   
4  https://www.foxnews.com/us/wis-hospital-intent...   coronavirus   

         date                                              title  \
0  2021-01-01  What Got Us Through 2020? For Many, It Was Hob...   
1  2020-12-31  New Year's Eve gatherings could accelerate COV...   
2  2020-12-31  Here’s how countries around the world are ring...   
3  2020-12-31  Wisconsin health worker 'deliberately spoiled ...   
4  2020-12-31  Wisconsin hospital employee 'intentionally' re...   

                   site           bias  \
0     NPR (Online News)   leaning-left   
1     CBS News (Online)   leaning-left   
2  New York Post (News)  leaning-right   
3 

In [5]:
# !python -m spacy download en_core_web_lg

nlp = spacy.load('en_core_web_lg')
nlp.add_pipe("textrank")

def clean_text(text):
    text2 = [token for token in text if not token.is_stop and not token.is_punct]
    text3 = [token.lemma_ for token in text2 if token.lemma_.lower() != "-PRON-"]
    return text3

In [7]:
# df['tokenized'] = df['page_text'].apply(lambda x: nlp(x))
# df['token_cleaned'] = df['tokenized'].apply(lambda x: clean_text(x))
df.to_csv('cleaned_dataset.csv', index=False)
df = pd.read_csv('cleaned_dataset.csv')

In [16]:
df['simple_bias'] = 'center'
df.loc[(df['bias'] == 'leaning-left') | (df['bias'] == 'left'), 'simple_bias'] = 'left'
df.loc[(df['bias'] == 'leaning-right') | (df['bias'] == 'right'), 'simple_bias'] = 'right'

In [17]:
df

,url,topic,date,title,site,bias,page_text,tokenized,token_cleaned,simple_bias
0,https://www.npr.org/2021/01/01/952336030/what-...,general-news,2021-01-01,"What Got Us Through 2020? For Many, It Was Hob...",NPR (Online News),leaning-left,"What Got Us Through 2020? For Many, It Was Hob...","What Got Us Through 2020? For Many, It Was Hob...","['get', '2020', 'Hobbies', 'relationship', '4'...",left
1,https://www.cbsnews.com/news/new-years-eve-gat...,general-news,2020-12-31,New Year's Eve gatherings could accelerate COV...,CBS News (Online),leaning-left,New Year's Eve gatherings could accelerate COV...,New Year's Eve gatherings could accelerate COV...,"['New', 'Year', 'Eve', 'gathering', 'accelerat...",left
2,https://nypost.com/2020/12/31/heres-how-countr...,general-news,2020-12-31,Here’s how countries around the world are ring...,New York Post (News),leaning-right,Here’s how countries around the world are ring...,Here’s how countries around the world are ring...,"['country', 'world', 'ring', '2021', '\n', 'wo...",right
3,https://www.theguardian.com/world/2020/dec/31/...,coronavirus,2020-12-31,Wisconsin health worker 'deliberately spoiled ...,The Guardian,left,Police in Wisconsin said on Thursday evening t...,Police in Wisconsin said on Thursday evening t...,"['Police', 'Wisconsin', 'say', 'Thursday', 'ev...",left
4,https://www.foxnews.com/us/wis-hospital-intent...,coronavirus,2020-12-31,Wisconsin hospital employee 'intentionally' re...,Fox News Digital,right,A Wisconsin-based hospital on Wednesday said a...,A Wisconsin-based hospital on Wednesday said a...,"['Wisconsin', 'base', 'hospital', 'Wednesday',...",right
...,...,...,...,...,...,...,...,...,...,...
10827,https://www.foxnews.com/world/bolivian-preside...,world,2024-06-27,"Bolivian president survives failed coup, calls...",Fox News Digital,right,Bolivian President Luis Arce announced three n...,Bolivian President Luis Arce announced three n...,"['bolivian', 'President', 'Luis', 'Arce', 'ann...",right
10828,https://apnews.com/article/biden-trump-ad-camp...,2024-presidential-election,2024-06-17,Biden’s campaign announces a $50 million adver...,Associated Press,left,Biden’s campaign announces a $50 million adver...,Biden’s campaign announces a $50 million adver...,"['Biden', 'campaign', 'announce', '$', '50', '...",left
10829,https://www.foxnews.com/politics/biden-campaig...,2024-presidential-election,2024-06-17,Biden campaign targets 'convicted felon' Trump...,Fox News Digital,right,The Biden campaign released a new ad Monday mo...,The Biden campaign released a new ad Monday mo...,"['Biden', 'campaign', 'release', 'new', 'ad', ...",right
10830,https://abcnews.go.com/Politics/1st-biden-trum...,2024-presidential-election,2024-06-17,1st Biden-Trump debate will include microphone...,ABC News (Online),leaning-left,1st Biden-Trump debate will include microphone...,1st Biden-Trump debate will include microphone...,"['1st', 'Biden', 'Trump', 'debate', 'include',...",left


In [81]:
corpus = st.CorpusFromPandas(df,
                             category_col='simple_bias',
                             text_col='token_cleaned',
                             nlp=nlp).build()

In [ ]:
corpus = corpus.remove_terms(['�', '� �', "' �", '\\n click', 'app \\n', '\\n �', '\\n \\n'])

In [102]:
term_freq_df = corpus.get_term_freq_df()
term_freq_df['Right Score'] = corpus.get_scaled_f_scores('right')
term_freq_df['Left Score'] = corpus.get_scaled_f_scores('left')
term_freq_df['Center Score'] = corpus.get_scaled_f_scores('center')

In [103]:
pprint(list(term_freq_df.sort_values(by='Right Score', ascending=False).index[:25]))

['fox news',
 'click fox',
 "' examiner",
 'news app',
 'click',
 'examiner',
 '\\n fox',
 'news digital',
 'click read',
 'tell fox',
 "examiner '",
 'fox',
 'press contribute',
 "fox '",
 'read washington',
 'digital',
 'app',
 "digital '",
 "' news",
 'r',
 "r '",
 "' r",
 'contribute report',
 't',
 'd']


In [104]:
pprint(list(term_freq_df.sort_values(by='Left Score', ascending=False).index[:25]))

["' ap",
 "' photo",
 "ap '",
 'ap',
 'tell cnn',
 'photo',
 '\\n cnn',
 'cnn',
 "cnn '",
 'gaza',
 'nbc',
 "nbc '",
 'official say',
 "2022 '",
 'file',
 "' ukraine",
 'oil',
 'ukrainian',
 "ukrainian '",
 'palestinian',
 'lawyer',
 'saturday',
 "kennedy '",
 'kennedy',
 'adviser']


In [105]:
pprint(list(term_freq_df.sort_values(by='Center Score', ascending=False).index[:25]))

["ms '",
 'ms',
 'mr trump',
 '\\n mr',
 'mr',
 "mr '",
 'bbc',
 "' mr",
 "key '",
 'tell bbc',
 '\\n key',
 'forbes',
 'defence',
 "uk '",
 'uk',
 'programme',
 'border guard',
 'content \\n',
 '£',
 '\\n allow',
 "allow '",
 "' content",
 "£ '",
 'topline',
 "forbes '"]


In [110]:
term_freq_df.to_csv('term_freq_df.csv')

In [107]:
html = st.produce_scattertext_explorer(corpus,
                                       category='left',
                                       category_name = 'Left',
                                       not_categories=['right'],
                                       not_category_name='Right',
                                       width_in_pixels=1000,
                                       minimum_term_frequency=1000,
                                       pmi_threshold_coefficient=0)
open("Bias_Visualization.html", 'wb').write(html.encode('utf-8'))

52688238

In [108]:
html = st.produce_scattertext_explorer(corpus,
                                       category='left',
                                       category_name = 'Left',
                                       not_categories=['right'],
                                       not_category_name='Right',
                                       width_in_pixels=1000,
                                       minimum_term_frequency=1000,
                                       pmi_threshold_coefficient=0,
                                       transform=st.Scalers.dense_rank)
open("Bias_Visualization_2.html", 'wb').write(html.encode('utf-8'))

52681972